In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

PATH = "/content/drive/MyDrive/VAT Project/Messy_VAT_5Years.xlsx"
OUT  = "/content/drive/MyDrive/VAT Project/VAT_Cleaned_5Years.xlsx"

raw = pd.read_excel(PATH, sheet_name=None)
purchases = raw["Purchases"].copy()
sales     = raw["Sales"].copy()

def clean_cols(df):
    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))
    return df
purchases, sales = clean_cols(purchases), clean_cols(sales)

for df in (purchases, sales):
    df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True, format="mixed")

def tidy_text(s):
    return s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()

purchases["supplier_name"] = tidy_text(purchases["supplier_name"]).replace({
    "Etislat": "Etisalat", "Etisalat Telecom": "Etisalat", "Aramexx": "Aramex",
    "A.D.D.C": "Abu Dhabi Distribution Company", "Addc": "Abu Dhabi Distribution Company",
    "Abu Dhabi Distrbution Company": "Abu Dhabi Distribution Company",
    "Eitc Du": "Du Telecom", "Sharaf Stationary": "Sharaf Stationery",
}).replace({"Tamm": "TAMM", "Tas-Heel": "Tas-heel"})

sales["customer_name"] = tidy_text(sales["customer_name"])
sales["service"] = (tidy_text(sales["service"])
                    .str.replace("Typng", "Typing", regex=False)
                    .replace({"Emirates Id Typing": "Emirates ID Typing",
                              "Pro Services": "PRO Services",
                              "Tamm Transaction": "TAMM Transaction",
                              "Tas-Heel Transaction": "Tas-heel Transaction"}))

def clean_amount(s):
    s = s.astype("string").str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(s, errors="coerce").astype("float64")

def flag_outliers(df, col):
    flag = pd.Series("ok", index=df.index)
    for _, idx in df.groupby(col).groups.items():
        v = df.loc[idx, "amount_aed"]
        q1, q3 = v.quantile([0.25, 0.75]); hi = q3 + 1.5 * (q3 - q1)
        flag.loc[v[v > hi].index] = "review"
    return flag

for df in (purchases, sales):
    df["amount_aed"] = clean_amount(df["amount_aed"]).abs()
purchases["amount_flag"] = flag_outliers(purchases, "category")
sales["amount_flag"]     = flag_outliers(sales, "service")

for df in (purchases, sales):
    df.loc[df["amount_flag"] == "review", "amount_aed"] = np.nan
    df["vat_5_aed"] = (df["amount_aed"] * 0.05).round(2)
    df["total_aed"] = (df["amount_aed"] + df["vat_5_aed"]).round(2)
    df.dropna(subset=["amount_aed", "date"], inplace=True)

purchases["supplier_name"] = purchases["supplier_name"].fillna("Unknown Supplier")
sales["customer_name"]     = sales["customer_name"].fillna("Unknown Customer")

purchases = purchases.drop_duplicates().drop(columns=["amount_flag"])
sales     = sales.drop_duplicates().drop(columns=["amount_flag"])

for name, df in [("Purchases", purchases), ("Sales", sales)]:
    assert df["amount_aed"].notna().all()
    assert df["date"].between("2020-01-01", "2024-12-31").all()
    assert (df["amount_aed"] > 0).all()
    assert np.allclose(df["vat_5_aed"], (df["amount_aed"] * 0.05).round(2))
    assert not df.duplicated().any()
    print(f"✓ {name} passed — {df.shape[0]} rows")

with pd.ExcelWriter(OUT) as w:
    purchases.to_excel(w, sheet_name="Purchases", index=False)
    sales.to_excel(w, sheet_name="Sales", index=False)
print("Saved:", OUT)

Mounted at /content/drive
✓ Purchases passed — 420 rows
✓ Sales passed — 5793 rows
Saved: /content/drive/MyDrive/VAT Project/VAT_Cleaned_5Years.xlsx
